A decorator = function transformation layer  
original function → wrapped function → enhanced behavior

In [1]:
def dec1(func):
    def wrapper():
        print("dec1 before")
        func()
        print("dec1 after")
    return wrapper

def dec2(func):
    def wrapper():
        print("dec2 before")
        func()
        print("dec2 after")
    return wrapper

@dec1
@dec2
def greet():
    print("hello")

greet()

dec1 before
dec2 before
hello
dec2 after
dec1 after


Decorators apply bottom → up  
Execution happens top → down

Class-Based Decorators   
Decorators don’t have to be functions.

### Class-Based Decorator — Notes

#### Overview
- A **class-based decorator** uses a class instead of a function to wrap another function
- Useful when you need:
  - State (store values across calls)
  - More control
  - Reusability

---

#### Basic Idea

- `__init__` → runs once when decorator is applied
- `__call__` → runs every time the function is called

---

#### Basic Example

```python
class MyDecorator:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        print("Before function call")
        result = self.func(*args, **kwargs)
        print("After function call")
        return result


@MyDecorator
def say_hello():
    print("Hello!")

say_hello()

Flow  
@MyDecorator → MyDecorator(say_hello) → calls __init__  
say_hello() → actually calls __call__

In [2]:
class Logger:

    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        print("Calling function")
        return self.func(*args, **kwargs)

@Logger
def greet():
    print("Hello")

greet()

Calling function
Hello


Why useful?  
Stateful decorators.  
Better structure for complex logic

In [ ]:
# Stateful Decorators
def counter(func):

    count = 0

    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        print(f"Called {count} times")
        return func(*args, **kwargs)

    return wrapper

In [ ]:
#caching decorator

def cache(func):
    memo = {}

    def wrapper(x):
        if x in memo:
            return memo[x]
        result = func(x)
        memo[x] = result
        return result

    return wrapper

In [4]:
# Decorators for Classes
def add_method(cls):
    cls.new_method = lambda self: "Hello"
    return cls

@add_method
class A:
    pass

print(A().new_method())


Hello


In [ ]:
# Decorator + Context Manager Pattern
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.time()
    yield
    print(time.time() - start)

In [ ]:
# exec order
# @A
# @B
# def f():

# f = A(B(f))

In [7]:
# Example with State
class CallCounter:
    def __init__(self, func):
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Called {self.count} times")
        return self.func(*args, **kwargs)


@CallCounter
def greet():
    print("Hi")

greet()
greet()

Called 1 times
Hi
Called 2 times
Hi


In [ ]:
# decorator with arguments
# Key Points
    # Class must implement __call__
    # Can store internal state (unlike simple functions)
    # Can be nested for arguments
class Repeat:
    def __init__(self, times):
        self.times = times

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            for _ in range(self.times):
                func(*args, **kwargs)
        return wrapper


@Repeat(3)
def hello():
    print("Hello")

hello()

Hello
Hello
Hello


| Feature     | Function Decorator | Class Decorator  |
| ----------- | ------------------ | ---------------- |
| Simplicity  | Easy               | Slightly complex |
| State       | Limited            | Easy             |
| Readability | High               | Medium           |
| Use case    | Simple wrapping    | Advanced control |

### When to Use
- When you need to track state (count, cache, metrics)
- When logic becomes complex
- When decorator needs configuration + behavior
### Summary
- Class-based decorators = more powerful + stateful
- Use for advanced scenarios
- For simple cases → prefer function decorators

In [9]:
# Stateful Decorator with Arguments
class LimitCalls:
    def __init__(self, max_calls):
        self.max_calls = max_calls

    def __call__(self, func):
        count = 0

        def wrapper(*args, **kwargs):
            nonlocal count
            if count >= self.max_calls:
                print("Limit reached")
                return
            count += 1
            return func(*args, **kwargs)

        return wrapper


@LimitCalls(2)
def hello():
    print("Hello")

hello()
hello()
hello()

# this happends
# hello = LimitCalls(2)(hello)

# Why not store count in class?
# Because:
    # Then it would be shared across ALL functions using same instance
    # Closure gives function-specific state

Hello
Hello
Limit reached


| Feature             | Normal Class Decorator | With Arguments (Stateful) |
| ------------------- | ---------------------- | ------------------------- |
| Syntax              | `@Decorator`           | `@Decorator(args)`        |
| `__init__` receives | function               | config (args)             |
| `__call__` receives | function call args     | function                  |
| Extra layer         | ❌ No                   | ✅ Yes                     |
| Wrapper needed      | ❌ No (optional)        | ✅ Yes                     |
| State handling      | via class              | via closure or class      |
| Complexity          | Simple                 | Advanced                  |


In [10]:
# 🔁 Mental Model
# Normal Decorator
    # @Decorator
    #    ↓
    # Decorator(func)
    #    ↓
    # object replaces func

# With Arguments
    # @Decorator(args)
    #    ↓
    # Decorator(args) → instance
    #    ↓
    # instance(func)
    #    ↓
    # wrapper returned
    #    ↓
    # func = wrapper

Docrator for class  
MyClass = decorator(MyClass)
- The class is passed to decorator
- Decorator returns a modified class (or same class)

In [ ]:
def add_method(cls):
    def greet(self):
        return "Hello from added method"
    
    cls.greet = greet
    return cls


@add_method
class Person:
    pass


p = Person()
print(p.greet())

# What Happens
#     Person class is created
#     Passed to add_method
#     Method greet is added dynamically
#     Modified class is returned

Hello from added method


In [ ]:
# Modifying Class Attributes
def add_attribute(cls):
    cls.role = "User"
    return cls


@add_attribute
class Employee:
    pass


e = Employee()
print(e.role)

User


In [13]:
class AddInitMessage:
    def __init__(self, cls):
        self.cls = cls

    def __call__(self, *args, **kwargs):
        print("Creating instance...")
        return self.cls(*args, **kwargs)


@AddInitMessage
class User:
    def __init__(self, name):
        self.name = name


u = User("Nishant")

Creating instance...


In [14]:
# with arguments
def add_tag(tag):
    def decorator(cls):
        cls.tag = tag
        return cls
    return decorator


@add_tag("premium")
class Product:
    pass


print(Product.tag)

premium


### Important Notes
    - Class decorators run once at definition time
    - They modify the class before instances are created
### Be careful with:
    - method wrapping (closure issues)
    - overriding existing methods 

In [15]:
# Decorator+contextmanager

import time
from contextlib import ContextDecorator

class Timer(ContextDecorator):
    def __enter__(self):
        self.start = time.time()
        print("Start")
        return self

    def __exit__(self, exc_type, exc, tb):
        print("End")
        print("Time:", time.time() - self.start)


# ✅ As decorator
@Timer()
def work():
    time.sleep(1)


# ✅ As context manager
with Timer():
    time.sleep(1)

work()

Start
End
Time: 1.00537109375
Start
End
Time: 1.0027410984039307
